In [10]:
import yfinance as yf
import pandas as pd
import numpy as np
import ta
import os
import time
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

In [2]:
# Assume the project root is the parent of src/
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
data_dir = os.path.join(project_root, "data", "raw")
os.makedirs(data_dir, exist_ok=True)

In [ ]:
tickers = ["AAPL", "MSFT", "NVDA", "AMZN", "JNJ", "JPM", "XOM", "CAT", "PG", "NEE"]
ticker_data = pd.DataFrame()


for ticker in tickers:
    data = yf.download(ticker, start='2020-01-01', end='2025-01-01', auto_adjust=True)
    #Only keep Adjusted Close and Volume
    features = data[['Close','Volume']].copy()
    features.columns = [f"{ticker}_Close", f"{ticker}_Volume"]
    ticker_data = pd.concat([ticker_data, features], axis=1)
    time.sleep(1)  #Avoiding API Limits

ticker_data.head()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


,AAPL_Close,AAPL_Volume,MSFT_Close,MSFT_Volume,NVDA_Close,NVDA_Volume,AMZN_Close,AMZN_Volume,JNJ_Close,JNJ_Volume,JPM_Close,JPM_Volume,XOM_Close,XOM_Volume,CAT_Close,CAT_Volume,PG_Close,PG_Volume,NEE_Close,NEE_Volume
Date,,,,,,,,,,,,,,,,,,,,
2020-01-02,72.538513,135480400,152.791153,22622100,5.971410,237536000,94.900497,80580000,124.072983,5777000,119.573372,10803700,54.131073,12456400,132.874985,3311900,106.273247,8130800,51.782585,7884800
2020-01-03,71.833298,146322800,150.888611,21116200,5.875832,205384000,93.748497,75288000,122.636490,5752400,117.995430,10386800,53.695885,17386900,131.030060,3100600,105.558487,7970500,52.151508,7097200
2020-01-06,72.405678,118387200,151.278641,20813700,5.900473,262636000,95.143997,81236000,122.483498,7731300,117.901604,10259000,54.108177,20081900,130.941803,2549600,105.704895,6674400,52.411919,5518800
2020-01-07,72.065140,108872000,149.899292,21634100,5.971909,314856000,95.343002,80898000,123.231468,7382900,115.897224,10531300,53.665344,17387700,129.211655,2841900,105.050415,7583400,52.366341,6653200
2020-01-08,73.224411,132079200,152.286972,27746500,5.983109,277108000,94.598503,70160000,123.214493,6605800,116.801315,9695300,52.856052,15137700,130.359177,2153200,105.498222,5385100,52.342468,5936000


In [4]:
returns = ticker_data[[col for col in ticker_data.columns if "Close" in col]].pct_change().dropna()
volume = ticker_data[[col for col in ticker_data.columns if "Volume" in col]].iloc[1:]
ml_data = pd.concat([returns, volume], axis=1)
print(ml_data.head())

            AAPL_Close  MSFT_Close  NVDA_Close  AMZN_Close  JNJ_Close  \
Date                                                                    
2020-01-03   -0.009722   -0.012452   -0.016006   -0.012139  -0.011578   
2020-01-06    0.007968    0.002585    0.004194    0.014886  -0.001248   
2020-01-07   -0.004703   -0.009118    0.012107    0.002092   0.006107   
2020-01-08    0.016086    0.015929    0.001876   -0.007809  -0.000138   
2020-01-09    0.021241    0.012493    0.010983    0.004799   0.002966   

            JPM_Close  XOM_Close  CAT_Close  PG_Close  NEE_Close  AAPL_Volume  \
Date                                                                            
2020-01-03  -0.013196  -0.008040  -0.013885 -0.006726   0.007124    146322800   
2020-01-06  -0.000795   0.007678  -0.000674  0.001387   0.004993    118387200   
2020-01-07  -0.017000  -0.008184  -0.013213 -0.006192  -0.000870    108872000   
2020-01-08   0.007801  -0.015080   0.008881  0.004263  -0.000456    132079200   
20

In [4]:
path = os.path.join(data_dir,"ML_DATA.csv")
ml_data = pd.read_csv(path)

In [ ]:
train_data = pd.read_csv(path, parse_dates=["Date"])
train_data = train_data.sort_values("Date").reset_index(drop=True)
train_data

,Date,AAPL_Close,MSFT_Close,NVDA_Close,AMZN_Close,JNJ_Close,JPM_Close,XOM_Close,CAT_Close,PG_Close,...,AAPL_Volume,MSFT_Volume,NVDA_Volume,AMZN_Volume,JNJ_Volume,JPM_Volume,XOM_Volume,CAT_Volume,PG_Volume,NEE_Volume
0,2020-01-03,-0.009722,-0.012452,-0.016006,-0.012139,-0.011578,-0.013196,-0.008040,-0.013885,-0.006726,...,146322800,21116200,205384000,75288000,5752400,10386800,17386900,3100600,7970500,7097200
1,2020-01-06,0.007968,0.002585,0.004194,0.014886,-0.001248,-0.000795,0.007678,-0.000674,0.001387,...,118387200,20813700,262636000,81236000,7731300,10259000,20081900,2549600,6674400,5518800
2,2020-01-07,-0.004703,-0.009118,0.012107,0.002092,0.006107,-0.017000,-0.008184,-0.013213,-0.006192,...,108872000,21634100,314856000,80898000,7382900,10531300,17387700,2841900,7583400,6653200
3,2020-01-08,0.016086,0.015929,0.001876,-0.007809,-0.000138,0.007801,-0.015080,0.008881,0.004263,...,132079200,27746500,277108000,70160000,6605800,9695300,15137700,2153200,5385100,5936000
4,2020-01-09,0.021241,0.012493,0.010983,0.004799,0.002966,0.003651,0.007656,-0.002505,0.010938,...,170108400,21385000,255112000,63346000,6112700,9469000,14811800,2272500,5944000,6958000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1252,2024-12-24,0.011478,0.009374,0.003938,0.017729,0.003993,0.016444,0.000941,0.005966,0.004937,...,23234700,7164500,105157000,15007500,3164100,3729100,7807000,886000,2460800,3001800
1253,2024-12-26,0.003176,-0.002777,-0.002068,-0.008732,-0.001851,0.003425,0.000846,-0.001224,0.007222,...,27237100,8194200,116205600,16146700,4656300,4451800,9652400,1097900,3629400,4503800
1254,2024-12-27,-0.013242,-0.017302,-0.020868,-0.014534,-0.003640,-0.008102,-0.000094,-0.006156,-0.003702,...,42355300,18117700,170582600,27367100,5588300,5730200,11943900,1245800,4367900,5458100
1255,2024-12-30,-0.013263,-0.013240,0.003503,-0.010950,-0.011789,-0.007671,-0.006762,-0.005070,-0.014393,...,35557500,13158700,167734700,28321200,6268700,5723800,11080800,1422200,4354500,8399000


In [ ]:
tickers = ["AAPL", "MSFT", "NVDA", "AMZN", "JNJ", "JPM", "XOM", "CAT", "PG", "NEE"]
train_data["Market_Avg"] = train_data[[f"{t}_Close" for t in tickers]].mean(axis=1)


for t in tickers:
    r = train_data[f"{t}_Close"] 
    v = train_data[f"{t}_Volume"]
    
    #Lagged returns
    train_data[f"{t}_Lag1"] = r.shift(1)
    train_data[f"{t}_Lag5"] = r.shift(5)
    train_data[f"{t}_Lag10"] = r.shift(10)
    
    #Rolling volatility
    train_data[f"{t}_Volatility5"] = r.rolling(5).std()
    train_data[f"{t}_Volatility20"] = r.rolling(20).std()
    
    #Rolling mean
    train_data[f"{t}_Mean5"] = r.rolling(5).mean()
    train_data[f"{t}_Mean20"] = r.rolling(20).mean()
    
    #Rolling correlation with average of all stocks
    train_data[f"{t}_CrossCorr5"] = r.rolling(5).corr(train_data["Market_Avg"])
    
    #Volume trend
    train_data[f"{t}_Vol_Mean5"] = v.rolling(5).mean()
    train_data[f"{t}_Vol_Change"] = v.pct_change()

#Drop rows with NaN
df = train_data.dropna().reset_index(drop=True)


/var/folders/_h/d7dp0dbd3hx5zl2d6xdvg3w40000gn/T/ipykernel_3268/3954225498.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_data[f"{t}_Mean20"] = r.rolling(20).mean()
/var/folders/_h/d7dp0dbd3hx5zl2d6xdvg3w40000gn/T/ipykernel_3268/3954225498.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_data[f"{t}_CrossCorr5"] = r.rolling(5).corr(train_data["Market_Avg"])
/var/folders/_h/d7dp0dbd3hx5zl2d6xdvg3w40000gn/T/ipykernel_3268/3954225498.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually th

,Date,AAPL_Close,MSFT_Close,NVDA_Close,AMZN_Close,JNJ_Close,JPM_Close,XOM_Close,CAT_Close,PG_Close,...,NEE_Lag1,NEE_Lag5,NEE_Lag10,NEE_Volatility5,NEE_Volatility20,NEE_Mean5,NEE_Mean20,NEE_CrossCorr5,NEE_Vol_Mean5,NEE_Vol_Change
0,2020-01-31,-0.044339,-0.014759,-0.038159,0.073791,-0.009909,-0.025977,-0.041210,-0.029696,-0.010560,...,0.015691,0.013996,0.002743,0.009649,0.006645,0.003412,0.005881,0.056809,7363120.0,0.190873
1,2020-02-03,-0.002746,0.024379,0.016496,-0.002250,0.008733,0.007631,-0.022376,-0.012029,0.003932,...,-0.006409,0.011451,0.004479,0.009032,0.007080,0.000085,0.005266,0.439302,7835600.0,0.464972
2,2020-02-04,0.033013,0.032916,0.028294,0.022687,0.009523,0.014396,-0.012514,0.028820,0.004396,...,-0.005183,-0.003486,0.013773,0.009777,0.007722,-0.000919,0.004591,0.196886,8279600.0,-0.050574
3,2020-02-05,0.008155,-0.001222,0.014688,-0.004781,0.015765,0.017000,0.046023,0.029436,0.009152,...,-0.008508,-0.000188,0.011717,0.010618,0.007650,0.000759,0.005044,0.334465,8811280.0,-0.173236
4,2020-02-06,0.011697,0.020734,0.013918,0.005079,-0.002987,0.000146,-0.013550,-0.001382,0.002602,...,0.008203,0.015691,0.000693,0.007709,0.007542,-0.001165,0.005371,0.326761,9120400.0,-0.091429
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1233,2024-12-24,0.011478,0.009374,0.003938,0.017729,0.003993,0.016444,0.000941,0.005966,0.004937,...,0.011441,-0.007144,-0.007497,0.019919,0.012527,0.001924,-0.002417,0.887457,12857200.0,-0.600240
1234,2024-12-26,0.003176,-0.002777,-0.002068,-0.008732,-0.001851,0.003425,0.000846,-0.001224,0.007222,...,0.005794,-0.024215,-0.002158,0.015284,0.012154,0.005285,-0.003342,0.940952,11258760.0,0.500366
1235,2024-12-27,-0.013242,-0.017302,-0.020868,-0.014534,-0.003640,-0.008102,-0.000094,-0.006156,-0.003702,...,-0.007406,-0.010635,-0.010273,0.013691,0.011507,0.006694,-0.004186,0.824342,9604040.0,0.211888
1236,2024-12-30,-0.013263,-0.013240,0.003503,-0.010950,-0.011789,-0.007671,-0.006762,-0.005070,-0.014393,...,-0.003593,0.027232,0.005463,0.007991,0.011418,0.000277,-0.004524,0.777851,5774340.0,0.538814


In [13]:
path = os.path.join(data_dir,"TRAINING_DATA.csv")
df.to_csv(path)

In [9]:
features = [col for col in df.columns if "AAPL" in col and "Close" not in col]
x = df[features]
y = df['AAPL_Close'].shift(-1)
x, y = x[:-1], y[:-1]

X_train, X_test, y_train, y_test = train_test_split(x,y, test_size = .2, shuffle = False)

In [11]:
model = RandomForestRegressor(n_estimators = 200, random_state = 42)
model.fit(X_train, y_train)


preds = model.predict(X_test)
mse = mean_squared_error(y_test, preds)
print(f"AAPL Test MSE: {mse:.4f}")

AAPL Test MSE: 0.0002


In [12]:
y_train_pred = model.predict(X_train)
train_mse = mean_squared_error(y_train, y_train_pred)
print(train_mse)

6.725183945351574e-05
